# Lesson 3 — Attention (runnable)

Attention lets every word **look at every other word** and blend in the ones that matter.

Most explanations show this with *random* numbers, so the attention matrix is meaningless. We won't.
We'll hand-build vectors with **two features you can read** — how *animal-ish* and how *action-ish* each word is —
so when the model decides "dog should pay attention to cat", you'll see exactly **why**.

Three steps: **scores** (dot products) → **softmax** (a 100% attention budget) → **mix** (weighted blend).
Companion: [`03_attention.py`](../03_attention.py) · hand-worked example in [`03_walkthrough.md`](../03_walkthrough.md).

## Imports

In [ ]:
import torch                                # tensors
import torch.nn.functional as F             # F.softmax

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False)

## Meaningful vectors (no randomness!)

Our sentence is **"the dog chased cat"**. We describe each word with just **2 numbers**:

| feature | meaning |
|---|---|
| dim 0 | **animal-ness** — is this a creature? |
| dim 1 | **action-ness** — is this a doing-word? |

So `dog = [1.0, 0.1]` (very animal, barely action) and `chased = [0.1, 1.0]` (barely animal, very action).
These are made up by hand so we can follow the math — in a real model they'd be *learned* (that's Lesson 2).

In [ ]:
words = ["the", "dog", "chased", "cat"]
#                     animal  action
x = torch.tensor([[   0.2,    0.1 ],   # the     - filler word, low on both
                  [   1.0,    0.1 ],   # dog     - very animal
                  [   0.1,    1.0 ],   # chased  - very action
                  [   0.9,    0.1 ]])  # cat     - very animal (a lot like dog!)

print(f"{'word':<8}{'animal':>8}{'action':>8}")
for w, v in zip(words, x):
    print(f"{w:<8}{v[0]:>8.1f}{v[1]:>8.1f}")
print(f"\nSHAPE TRAIL:  {len(words)} words × {x.shape[1]} features  ->  x is {tuple(x.shape)}")

## Step 1 — Scores: how much should each word care about each other word?

The score for a pair is their **dot product** (multiply matching features, add them up).
`dog · cat = 1.0×0.9 + 0.1×0.1 = 0.91` — big, because **both are animals**.
`dog · chased = 1.0×0.1 + 0.1×1.0 = 0.20` — small, different kinds of words.

`x @ x.T` does all 16 pairs at once.

In [ ]:
scores = x @ x.T                              # (4 words) x (4 words) grid of dot products
print("Raw scores (bigger = more alike):\n")
print(f"{'':>8}" + "".join(f"{w:>8}" for w in words))
for i, w in enumerate(words):
    print(f"{w:>8}" + "".join(f"{scores[i,j]:>8.2f}" for j in range(len(words))))
print(f"\nSHAPE TRAIL:  x {tuple(x.shape)} @ x.T {tuple(x.T.shape)}  ->  scores {tuple(scores.shape)}  (every word vs every word)")
print("Look at the dog row: its biggest other-score is CAT. Animals recognise animals.")

## Step 2 — Softmax: turn scores into a 100% attention budget

Each word has 100% of attention to spend. **Softmax** turns a row of scores into percentages that add to 1,
giving the biggest scores the biggest share.

In [ ]:
attention = F.softmax(scores, dim=-1)         # each ROW now sums to 1.0
print("Attention weights (each row adds up to 1.00):\n")
print(f"{'':>8}" + "".join(f"{w:>8}" for w in words))
for i, w in enumerate(words):
    print(f"{w:>8}" + "".join(f"{attention[i,j]:>8.2f}" for j in range(len(words))))
print(f"\nrow sums = {attention.sum(dim=-1).tolist()}  (each spends exactly 100%)")

### Read it as a heatmap 🔥

Darker / fuller block = more attention. Now the pattern jumps out.

In [ ]:
def heatmap(M, rows, cols):
    shades = " .:-=+*#%@"                      # 10 shades from empty to full
    print(f"{'':>9}" + "".join(f"{c:>9}" for c in cols))
    for i, r in enumerate(rows):
        line = f"{r:>9}"
        for j in range(len(cols)):
            v = M[i, j].item()
            ch = shades[min(int(v * (len(shades) - 1)), len(shades) - 1)]
            line += f"  {v:4.2f}{ch}{ch} "
        print(line)

print("Attention heatmap (row = the word doing the looking):\n")
heatmap(attention, words, words)
print("\n• 'dog' row: most ink on dog & CAT  -> dog leans on the other animal.")
print("• 'chased' row: most ink on itself   -> the action word mostly minds its own business.")

## Step 3 — Mix: each word's new vector = weighted blend

Multiply the attention weights by the original vectors. Each word becomes a blend, weighted by who it looked at.

In [ ]:
new_x = attention @ x                         # blend
print("Vectors AFTER attention:\n")
print(f"{'word':<8}{'animal':>8}{'action':>8}")
for w, v in zip(words, new_x):
    print(f"{w:<8}{v[0]:>8.2f}{v[1]:>8.2f}")
print(f"\nSHAPE TRAIL:  attention {tuple(attention.shape)} @ x {tuple(x.shape)}  ->  new_x {tuple(new_x.shape)}  (same shape as we started!)")

### Watch ONE word absorb context

Let's track **dog** before and after — it should soak up a little extra animal-ness from cat.

In [ ]:
i = words.index("dog")
before = x[i]
after  = new_x[i]
print(f"dog BEFORE attention:  animal={before[0]:.2f}  action={before[1]:.2f}")
print(f"dog AFTER  attention:  animal={after[0]:.2f}  action={after[1]:.2f}")
print()
top = attention[i].argmax().item()
print(f"Who did 'dog' pay the MOST attention to?  ->  '{words[top]}'  ({attention[i,top]:.0%} of its budget)")
others = [(words[j], attention[i,j].item()) for j in range(len(words)) if j != i]
others.sort(key=lambda t: -t[1])
print("Rest of dog's attention:  " + ",  ".join(f"{w} {p:.0%}" for w, p in others))
print("\n'dog' kept most of itself but blended in CAT — it now carries 'I am near another animal'.")
print("That blended vector is what gets passed up to the next layer. THAT is attention.")

## Real attention: Q, K, V (one new idea at a time)

So far each word used the **same vector** for two different jobs: *"what am I looking for?"* and *"what do I offer?"*
Real attention splits those with three small learned matrices, so a word can **ask** for one thing and **advertise** another.

- **Q (query)** = what I'm looking for
- **K (key)** = what I advertise to others
- **V (value)** = what I hand over if picked

Each is just `x` multiplied by a learned matrix. Let's build them one at a time.

In [ ]:
d = x.shape[1]                                # feature size = 2
torch.manual_seed(3)
Wq = torch.randn(d, d)                        # in a real model these are LEARNED (L1's loop tunes them)
Wk = torch.randn(d, d)
Wv = torch.randn(d, d)

Q = x @ Wq                                    # each word's "question"
K = x @ Wk                                    # each word's "advertisement"
V = x @ Wv                                    # each word's "offering"
print(f"x  {tuple(x.shape)}   @ Wq {tuple(Wq.shape)}  ->  Q {tuple(Q.shape)}")
print(f"(same shape for K and V — every word still has {d} numbers, just re-cast for its 3 roles)")
print("\nQ (each word's query vector):")
for w, q in zip(words, Q):
    print(f"  {w:<8}{q.tolist()}")

### Why divide by √d?

Before softmax we scale the scores by `1/√d`. Why? As vectors get longer (more features), raw dot products grow
huge, and softmax of huge numbers becomes **too spiky** — one word grabs ~100% and the rest get ~0%, so the model
stops blending. Dividing by `√d` keeps scores in a sane range. See it on plain numbers:

In [ ]:
small = torch.tensor([2.0, 1.0, 0.0])
big   = torch.tensor([20.0, 10.0, 0.0])       # same shape, just 10x bigger
print("softmax([2, 1, 0])    =", [f"{p:.2f}" for p in F.softmax(small, dim=0)], " <- nicely shared")
print("softmax([20, 10, 0])  =", [f"{p:.2f}" for p in F.softmax(big,   dim=0)], " <- one word hogs it all")
print("\nThat 'too spiky' problem is what 1/sqrt(d) prevents. Now the full thing:\n")

scores2   = Q @ K.T / (d ** 0.5)              # scaled scores
attention2 = F.softmax(scores2, dim=-1)
output     = attention2 @ V                    # blend the VALUES
print("Scaled dot-product attention output:")
for w, v in zip(words, output):
    print(f"  {w:<8}{v.tolist()}")
print(f"\nSHAPE TRAIL:  Q{tuple(Q.shape)}·K^T -> scores {tuple(scores2.shape)} -> softmax -> @V -> output {tuple(output.shape)}")

**That's the whole formula** at the heart of GPT, BERT and PRAGMA:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

Everything else is repeating it (multi-head, many layers).

## Things to try

1. In the meaningful-vectors cell, make `cat = [0.1, 1.0]` (turn the cat into an action word!). Re-run — does `dog` still attend to `cat`?
2. Add a 5th word `mouse = [0.8, 0.1]` (another animal). Who attends to whom now?
3. In the heatmap, find the single biggest off-diagonal cell. Which two words is it?
4. Delete the `/ (d ** 0.5)` scaling in the last cell. Print `attention2` — did it get spikier?